In [1]:
!pip install rouge

Defaulting to user installation because normal site-packages is not writeable
  Using cached rouge-1.0.1-py3-none-any.whl.metadata (4.1 kB)
Using cached rouge-1.0.1-py3-none-any.whl (13 kB)



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached tensorflow-2.18.0-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
  Using cached tensorflow_intel-2.18.0-cp312-cp312-win_amd64.whl.metadata (4.9 kB)
  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.1.24-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached grpcio-1.70.0-cp312-cp312-win_amd64.whl.metadata (4.0 kB)
  Using cached tensorboard-2.18.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached keras-3.8.0-py3-none-any.whl.metadata (5.8 kB)
  Us

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 3.4 MB/s eta 0:00:01
   ---------------------------------- ----- 1.3/1.5 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 3.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


# CNN (Convolution Neural Network)

In [4]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from rouge_score import rouge_scorer

# Load Dataset
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# Sentence Split Function
def split_sentences(text):
    return re.split(r'(?<=[.!?]) +', text)

# Create Sentence-Level Dataset
sentences = []
labels = []

for _, row in data.iterrows():
    abstract = row['Abstract']
    summary = row['Summarized Abstract']

    abs_sentences = split_sentences(abstract)

    for sent in abs_sentences:
        sentences.append(sent)

        # Label = 1 if sentence appears in summary
        if sent.strip() in summary:
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

# Tokenization
num_words = 5000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# Build CNN Model
model = Sequential([
    Embedding(num_words, 128, input_length=max_len),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train
model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

# Generate Summary Function
def generate_summary(text, top_k=3):
    sents = split_sentences(text)

    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad).flatten()

    # Select top sentences
    top_indices = scores.argsort()[-top_k:][::-1]
    selected = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected)

# Evaluate with ROUGE
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

for _, row in data.iterrows():
    pred = generate_summary(row['Abstract'])
    ref = row['Summarized Abstract']

    scores = scorer.score(ref, pred)

    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougeL.append(scores['rougeL'].fmeasure)

print("ROUGE-1:", np.mean(rouge1))
print("ROUGE-2:", np.mean(rouge2))
print("ROUGE-L:", np.mean(rougeL))

C:\Users\sonal\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 95ms/step - accuracy: 0.7730 - loss: 0.6026 - val_accuracy: 1.0000 - val_loss: 0.1716
Epoch 2/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9966 - loss: 0.1126 - val_accuracy: 1.0000 - val_loss: 0.0183
Epoch 3/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.9978 - loss: 0.0296 - val_accuracy: 1.0000 - val_loss: 0.0115
Epoch 4/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9919 - loss: 0.0724 - val_accuracy: 1.0000 - val_loss: 0.0093
Epoch 5/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9989 - loss: 0.0170 - val_accuracy: 1.0000 - val_loss: 0.0080
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0

In [10]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from rouge_score import rouge_scorer

# Load Dataset
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# Sentence Split Function
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

# Create Sentence-Level Dataset
sentences = []
labels = []

for _, row in data.iterrows():
    abstract = row['Abstract']
    summary = row['Summarized Abstract']

    abs_sentences = split_sentences(abstract)

    for sent in abs_sentences:
        sentences.append(sent)

        # Label 1 if sentence appears in summary
        if sent.strip() in summary:
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

# Tokenization
num_words = 10000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# Build CNN Model
model = Sequential([
    Embedding(num_words, 128, input_length=max_len),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train Model
model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# Generate Summary Function
def generate_summary(text, top_k=3):
    sents = split_sentences(text)

    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad).flatten()

    top_indices = scores.argsort()[-top_k:][::-1]
    selected_sentences = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected_sentences)

# Test Examples
for i in range(5):
    original = data.iloc[i]['Abstract']
    actual = data.iloc[i]['Summarized Abstract']
    predicted = generate_summary(original)

    print("\nExample", i+1)
    print("Actual Summary:", actual)
    print("Predicted Summary:", predicted)

# ROUGE Evaluation
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

for _, row in data.iterrows():
    pred = generate_summary(row['Abstract'])
    ref = row['Summarized Abstract']

    scores = scorer.score(ref, pred)

    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougeL.append(scores['rougeL'].fmeasure)

print("\nROUGE-1:", np.mean(rouge1))
print("ROUGE-2:", np.mean(rouge2))
print("ROUGE-L:", np.mean(rougeL))

C:\Users\sonal\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_4          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 101ms/step - accuracy: 0.6476 - loss: 0.6089 - val_accuracy: 1.0000 - val_loss: 0.0696
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.9990 - loss: 0.0393 - val_accuracy: 1.0000 - val_loss: 3.8312e-05
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.9961 - loss: 0.0460 - val_accuracy: 1.0000 - val_loss: 1.1451e-05
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.9968 - loss: 0.0297 - val_accuracy: 1.0000 - val_loss: 3.2016e-05
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.9992 - loss: 0.0071 - val_accuracy: 1.0000 - val_loss: 8.0928e-05
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.9990 - loss: 0.0087 - val_accuracy: 1.0000 - val_loss: 3.9962e-04
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.9985 - loss: 0.0107 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.9978 - loss: 0.0151 - val_accurac

## RNN (Recurrent Neural Network)

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.translate.bleu_score import corpus_bleu

# Load dataset
data = pd.read_csv('ClinicalData.csv')

# Preprocessing
data.dropna(inplace=True)  
data = data[['Abstract', 'Summarized Abstract']]  

# Split data into training (75%) and test (25%)
train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)

# Tokenization and padding for text data
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")  
tokenizer.fit_on_texts(train_data['Abstract'])
X_train = tokenizer.texts_to_sequences(train_data['Abstract'])
X_test = tokenizer.texts_to_sequences(test_data['Abstract'])
max_len = 100  
X_train = pad_sequences(X_train, maxlen=max_len, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=max_len, padding='post', truncating='post')

# Tokenize and pad the target data (Summarized Abstract)
target_tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")  
target_tokenizer.fit_on_texts(train_data['Summarized Abstract'])
y_train = target_tokenizer.texts_to_sequences(train_data['Summarized Abstract'])
y_test = target_tokenizer.texts_to_sequences(test_data['Summarized Abstract'])
max_summary_len = 100  
y_train = pad_sequences(y_train, maxlen=max_summary_len, padding='post', truncating='post')
y_test = pad_sequences(y_test, maxlen=max_summary_len, padding='post', truncating='post')

# IMPORTANT FIX: reshape for sparse_categorical_crossentropy
y_train = np.expand_dims(y_train, -1)
y_test = np.expand_dims(y_test, -1)

# Build and compile the RNN model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=5000, output_dim=100),
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(5000, activation='softmax'))
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=5, batch_size=16, validation_split=0.2)

# Generate predicted summaries for the test data
predicted_summaries = model.predict(X_test, verbose=0)

# Convert probability outputs to word indices
predicted_sequences = np.argmax(predicted_summaries, axis=-1)

# Convert sequences back to text
predicted_summaries_text = []
for seq in predicted_sequences:
    words = []
    for idx in seq:
        if idx != 0:
            word = target_tokenizer.index_word.get(idx, "")
            words.append(word)
    predicted_summaries_text.append(words)

actual_summaries_text = []
for seq in y_test:
    words = []
    for idx in seq.flatten():
        if idx != 0:
            word = target_tokenizer.index_word.get(idx, "")
            words.append(word)
    actual_summaries_text.append(words)

# Calculate ROUGE scores
from nltk.translate.bleu_score import corpus_bleu

# BLEU format
references = [[ref] for ref in actual_summaries_text]
hypotheses = predicted_summaries_text

rouge_1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0))
rouge_2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0))
rouge_l = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0))
rouge_lsum = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))

print("ROUGE-1:", rouge_1)
print("ROUGE-2:", rouge_2)
print("ROUGE-L:", rouge_l)
print("ROUGE-Lsum:", rouge_lsum)

Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 52s 7s/step - accuracy: 0.0557 - loss: 8.2844 - val_accuracy: 0.1175 - val_loss: 12.7256
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.2090 - loss: 10.8982 - val_accuracy: 0.1187 - val_loss: 7.6634
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.2074 - loss: 6.9929 - val_accuracy: 0.1187 - val_loss: 7.5844
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.2156 - loss: 6.2857 - val_accuracy: 0.1187 - val_loss: 7.2417
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 986ms/step - accuracy: 0.2135 - loss: 5.6565 - val_accuracy: 0.1187 - val_loss: 7.4476
ROUGE-1: 1.0559787469384396e-13
ROUGE-2: 3.977501057280357e-14
ROUGE-L: 1.83014177698399e-115
ROUGE-Lsum: 1.1223333942556292e-167


C:\Users\sonal\AppData\Roaming\Python\Python312\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\sonal\AppData\Roaming\Python\Python312\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [14]:
# RNN print actual summary and predicted summary

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load dataset
data = pd.read_csv('ClinicalData.csv')

# Preprocessing
data.dropna(inplace=True)
data = data[['Abstract', 'Summarized Abstract']]

# Split data into training (75%) and test (25%)
train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)

# TOKENIZATION 

vocab_size = 20000  # Increased vocabulary for clinical data

# Tokenization and padding for text data
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(train_data['Abstract'])

X_train = tokenizer.texts_to_sequences(train_data['Abstract'])
X_test = tokenizer.texts_to_sequences(test_data['Abstract'])

max_len = 100

X_train = pad_sequences(X_train, maxlen=max_len, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=max_len, padding='post', truncating='post')

# Tokenize and pad the target data (Summarized Abstract)
y_train = target_tokenizer.texts_to_sequences(train_data['Summarized Abstract'])
y_test = target_tokenizer.texts_to_sequences(test_data['Summarized Abstract'])

# IMPORTANT: summary length must match input length
y_train = pad_sequences(y_train, maxlen=max_len, padding='post', truncating='post')
y_test = pad_sequences(y_test, maxlen=max_len, padding='post', truncating='post')

# reshape for sparse loss
y_train = np.expand_dims(y_train, -1)
y_test = np.expand_dims(y_test, -1)
# MODEL

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=128),
    
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    tf.keras.layers.SimpleRNN(256, activation='elu', return_sequences=True),
    
    tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(vocab_size, activation='softmax')
    )
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=16,
    validation_split=0.2
)

# PREDICTION

predicted_probabilities = model.predict(X_test, verbose=0)

# Take most probable word at each timestep
predicted_sequences = np.argmax(predicted_probabilities, axis=-1)

# PRINT ACTUAL VS PREDICTED 

for i in range(5):

    # Decode actual summary
    actual_words = []
    for idx in y_test[i].flatten():
        if idx != 0:
            word = target_tokenizer.index_word.get(idx, "")
            actual_words.append(word)

    actual_summary = " ".join(actual_words)

    # Decode predicted summary
    predicted_words = []
    for idx in predicted_sequences[i]:
        if idx != 0:
            word = target_tokenizer.index_word.get(idx, "")
            predicted_words.append(word)

    predicted_summary = " ".join(predicted_words)

    print("\n")
    print(f"Example {i+1}")
    print("Actual Summary:")
    print(actual_summary)
    print("\nPredicted Summary:")
    print(predicted_summary)

Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 48s 14s/step - accuracy: 0.0555 - loss: 9.8543 - val_accuracy: 0.1175 - val_loss: 8.9562
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 4s/step - accuracy: 0.2098 - loss: 8.4861 - val_accuracy: 0.0312 - val_loss: 8.3196
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 4s/step - accuracy: 0.0315 - loss: 6.6615 - val_accuracy: 0.0437 - val_loss: 7.8617
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 4s/step - accuracy: 0.0402 - loss: 6.4120 - val_accuracy: 0.0437 - val_loss: 8.4022
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 4s/step - accuracy: 0.0397 - loss: 5.9730 - val_accuracy: 0.1213 - val_loss: 8.4159


Example 1
Actual Summary:
are the primary for the of and and models have in understanding these however the of a such as a should factors this scoping review aims to identify and the most factors as well as for of and

Predicted Summary:
the the the the the the the the


Example 2
Actual Summary:
the of clinical is a before more natural language processing models that high accuracy f

# LSTM (Long short-term memory)

In [18]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GlobalMaxPooling1D, Dense, Dropout
from rouge_score import rouge_scorer

# Load Dataset
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# Sentence Split Function
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

# Create Sentence-Level Dataset
sentences = []
labels = []

for _, row in data.iterrows():
    abstract = row['Abstract']
    summary = row['Summarized Abstract']

    abs_sentences = split_sentences(abstract)

    for sent in abs_sentences:
        sentences.append(sent)

        # Label = 1 if sentence appears in summary
        if sent.strip().lower() in summary.lower():
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

# Tokenization
num_words = 10000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# LSTM MODEL 
model = Sequential([
    Embedding(input_dim=num_words, output_dim=128),
    LSTM(128, return_sequences=True),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train
model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# Generate Summary 
def generate_summary(text, top_k=3):
    sents = split_sentences(text)

    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad, verbose=0).flatten()

    top_indices = scores.argsort()[-top_k:][::-1]
    selected = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected)

# ROUGE Evaluation 
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

test_sample = data.sample(frac=0.25, random_state=42)

for _, row in test_sample.iterrows():
    pred = generate_summary(row['Abstract'])
    ref = row['Summarized Abstract']

    scores = scorer.score(ref, pred)

    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougeL.append(scores['rougeL'].fmeasure)

print("ROUGE-1:", np.mean(rouge1))
print("ROUGE-2:", np.mean(rouge2))
print("ROUGE-L:", np.mean(rougeL))

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_19 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_5          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 229ms/step - accuracy: 0.7663 - loss: 0.6295 - val_accuracy: 1.0000 - val_loss: 0.1658
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 149ms/step - accuracy: 0.9927 - loss: 0.1163 - val_accuracy: 1.0000 - val_loss: 0.0101
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 145ms/step - accuracy: 0.9974 - loss: 0.0256 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 130ms/step - accuracy: 0.9990 - loss: 0.0100 - val_accuracy: 1.0000 - val_loss: 9.3375e-04
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 114ms/step - accuracy: 0.9990 - loss: 0.0077 - val_accuracy: 1.0000 - val_loss: 7.6396e-04
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - accuracy: 0.9978 - loss: 0.0167 - val_accuracy: 1.0000 - val_loss: 8.1198e-04
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - accuracy: 0.9961 - loss: 0.0238 - val_accuracy: 1.0000 - val_loss: 8.5198e-04
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.9992 - loss: 0.0056 - val_acc

In [20]:
#LSTM - PRINTING ACTUAL AND PREDICTED SUMMARIES TEXT

import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GlobalMaxPooling1D, Dense, Dropout

# Load Dataset
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# Sentence Split
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

sentences = []
labels = []

for _, row in data.iterrows():
    abs_text = row['Abstract']
    sum_text = row['Summarized Abstract']
    abs_sentences = split_sentences(abs_text)

    for sent in abs_sentences:
        sentences.append(sent)
        if sent.strip().lower() in sum_text.lower():
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

num_words = 10000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# LSTM MODEL
model = Sequential([
    Embedding(input_dim=num_words, output_dim=128),
    LSTM(128, return_sequences=True),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# Generate Summary
def generate_summary(text, top_k=3):
    sents = split_sentences(text)
    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad, verbose=0).flatten()

    top_indices = scores.argsort()[-top_k:][::-1]
    selected = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected)

for i in range(5):
    original = data.iloc[i]['Abstract']
    actual = data.iloc[i]['Summarized Abstract']
    predicted = generate_summary(original)

    print("\nExample", i+1)
    print("Actual Summary:", actual)
    print("Predicted Summary:", predicted)

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 15s 224ms/step - accuracy: 0.8589 - loss: 0.5176 - val_accuracy: 1.0000 - val_loss: 0.0090
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.9961 - loss: 0.0289 - val_accuracy: 1.0000 - val_loss: 6.1118e-04
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step - accuracy: 0.9974 - loss: 0.0218 - val_accuracy: 1.0000 - val_loss: 3.6657e-04
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 130ms/step - accuracy: 0.9978 - loss: 0.0229 - val_accuracy: 1.0000 - val_loss: 5.4610e-04
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 128ms/step - accuracy: 0.9974 - loss: 0.0173 - val_accuracy: 1.0000 - val_loss: 8.8774e-04
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 129ms/step - accuracy: 0.9988 - loss: 0.0093 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - accuracy: 0.9968 - loss: 0.0213 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 140ms/step - accuracy: 0.9961 - loss: 0.0279 - val_acc

# CNN + LSTM

In [21]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, GlobalMaxPooling1D, Dense, Dropout
from rouge_score import rouge_scorer

# Load Dataset
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# Sentence split
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

sentences = []
labels = []

for _, row in data.iterrows():
    abstract = row['Abstract']
    summary = row['Summarized Abstract']

    abs_sents = split_sentences(abstract)

    for sent in abs_sents:
        sentences.append(sent)
        if sent.strip().lower() in summary.lower():
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

num_words = 10000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# CNN + LSTM MODEL 
model = Sequential([
    Embedding(input_dim=num_words, output_dim=128),

    Conv1D(128, 5, activation='relu'),
    MaxPooling1D(2),

    LSTM(128, return_sequences=True),
    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# Generate Summary 
def generate_summary(text, top_k=3):
    sents = split_sentences(text)
    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad, verbose=0).flatten()
    top_indices = scores.argsort()[-top_k:][::-1]
    selected = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected)

# ROUGE Evaluation 
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

test_sample = data.sample(frac=0.25, random_state=42)

for _, row in test_sample.iterrows():
    pred = generate_summary(row['Abstract'])
    ref = row['Summarized Abstract']

    scores = scorer.score(ref, pred)

    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougeL.append(scores['rougeL'].fmeasure)

print("ROUGE-1:", np.mean(rouge1))
print("ROUGE-2:", np.mean(rouge2))
print("ROUGE-L:", np.mean(rougeL))

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 24s 177ms/step - accuracy: 0.8078 - loss: 0.4708 - val_accuracy: 1.0000 - val_loss: 0.0048
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.9961 - loss: 0.0324 - val_accuracy: 1.0000 - val_loss: 3.4101e-04
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.9985 - loss: 0.0138 - val_accuracy: 1.0000 - val_loss: 2.3835e-04
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.9990 - loss: 0.0073 - val_accuracy: 1.0000 - val_loss: 3.2641e-04
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.9961 - loss: 0.0322 - val_accuracy: 1.0000 - val_loss: 7.4792e-04
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.9961 - loss: 0.0348 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - accuracy: 0.9988 - loss: 0.0110 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.9982 - loss: 0.0164 - val_accuracy: 

In [26]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, LSTM, GlobalMaxPooling1D, Dense, Dropout
from rouge_score import rouge_scorer

# Load Dataset
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# SENTENCE SPLIT FUNCTION
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

sentences = []
labels = []

for _, row in data.iterrows():
    abstract = row['Abstract']
    summary = row['Summarized Abstract']

    abs_sentences = split_sentences(abstract)

    for sent in abs_sentences:
        sentences.append(sent)

        if sent.strip().lower() in summary.lower():
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

# TOKENIZATION
num_words = 10000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# CNN + LSTM MODEL
model = Sequential([
    Embedding(input_dim=num_words, output_dim=128),

    Conv1D(128, 5, activation='relu', padding='same'),
    LSTM(128, return_sequences=True),

    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# TRAIN MODEL
model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# SUMMARY GENERATION FUNCTION
def generate_summary(text, top_k=3):
    sents = split_sentences(text)

    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad, verbose=0).flatten()

    top_indices = scores.argsort()[-top_k:][::-1]
    selected = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected)

for i in range(5):
    original = data.iloc[i]['Abstract']
    actual = data.iloc[i]['Summarized Abstract']
    predicted = generate_summary(original)

    print("\nExample", i+1)
    print("Actual Summary:", actual)
    print("Predicted Summary:", predicted)

# ROUGE Evaluation
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

test_sample = data.sample(frac=0.25, random_state=42)

for _, row in test_sample.iterrows():
    pred = generate_summary(row['Abstract'])
    ref = row['Summarized Abstract']

    scores = scorer.score(ref, pred)

    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougeL.append(scores['rougeL'].fmeasure)

print("\nROUGE-1:", np.mean(rouge1))
print("ROUGE-2:", np.mean(rouge2))
print("ROUGE-L:", np.mean(rougeL))

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_29 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_14 (Conv1D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_21 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_8          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 27s 286ms/step - accuracy: 0.8676 - loss: 0.4980 - val_accuracy: 1.0000 - val_loss: 0.0077
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 142ms/step - accuracy: 0.9982 - loss: 0.0161 - val_accuracy: 1.0000 - val_loss: 3.7673e-04
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 121ms/step - accuracy: 0.9927 - loss: 0.0553 - val_accuracy: 1.0000 - val_loss: 3.1498e-04
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - accuracy: 0.9927 - loss: 0.0616 - val_accuracy: 1.0000 - val_loss: 4.5788e-04
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - accuracy: 0.9982 - loss: 0.0156 - val_accuracy: 1.0000 - val_loss: 6.2640e-04
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 113ms/step - accuracy: 0.9949 - loss: 0.0379 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 112ms/step - accuracy: 0.9990 - loss: 0.0075 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - accuracy: 0.9982 - loss: 0.0171 - val_acc

# biLSTM (Bidirectional LSTM)

In [27]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GlobalMaxPooling1D, Dense, Dropout
from rouge_score import rouge_scorer

# LOAD DATA 
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# SENTENCE SPLIT FUNCTION
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

sentences = []
labels = []

for _, row in data.iterrows():
    abstract = row['Abstract']
    summary = row['Summarized Abstract']
    abs_sentences = split_sentences(abstract)

    for sent in abs_sentences:
        sentences.append(sent)

        # Binary label (same as your CNN logic)
        if sent.strip().lower() in summary.lower():
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

# TOKENIZATION
num_words = 10000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# BIDIRECTIONAL LSTM MODEL
model = Sequential([
    Embedding(input_dim=num_words, output_dim=128),

    Bidirectional(LSTM(128, return_sequences=True)),

    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# TRAIN MODEL
model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# SUMMARY GENERATION FUNCTION
def generate_summary(text, top_k=3):
    sents = split_sentences(text)
    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad, verbose=0).flatten()

    top_indices = scores.argsort()[-top_k:][::-1]
    selected = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected)

# PRINT ACTUAL AND PREDICTED SUMMARIES
for i in range(5):
    original = data.iloc[i]['Abstract']
    actual = data.iloc[i]['Summarized Abstract']
    predicted = generate_summary(original)

    print("\nExample", i+1)
    print("Actual Summary:", actual)
    print("Predicted Summary:", predicted)

# ROUGE Evaluation
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

test_sample = data.sample(frac=0.25, random_state=42)

for _, row in test_sample.iterrows():
    pred = generate_summary(row['Abstract'])
    ref = row['Summarized Abstract']

    scores = scorer.score(ref, pred)

    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougeL.append(scores['rougeL'].fmeasure)

print("\nROUGE-1:", np.mean(rouge1))
print("ROUGE-2:", np.mean(rouge2))
print("ROUGE-L:", np.mean(rougeL))

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_30 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_9          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 23s 419ms/step - accuracy: 0.7763 - loss: 0.4835 - val_accuracy: 1.0000 - val_loss: 5.4574e-04
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 239ms/step - accuracy: 0.9988 - loss: 0.0148 - val_accuracy: 1.0000 - val_loss: 1.9784e-05
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 260ms/step - accuracy: 0.9927 - loss: 0.0585 - val_accuracy: 1.0000 - val_loss: 9.0559e-05
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 220ms/step - accuracy: 0.9995 - loss: 0.0040 - val_accuracy: 1.0000 - val_loss: 1.2880e-04
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 144ms/step - accuracy: 0.9992 - loss: 0.0082 - val_accuracy: 1.0000 - val_loss: 5.3458e-04
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 250ms/step - accuracy: 0.9992 - loss: 0.0067 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 148ms/step - accuracy: 0.9927 - loss: 0.0589 - val_accuracy: 1.0000 - val_loss: 0.0030
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 166ms/step - accuracy: 0.9985 - loss: 0.0116 - val

# CNN + RNN

In [29]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, LSTM, GlobalMaxPooling1D, Dense, Dropout
from rouge_score import rouge_scorer

# LOAD DATA
data = pd.read_csv("ClinicalData.csv")
data = data[['Abstract', 'Summarized Abstract']]
data.dropna(inplace=True)

# SENTENCE SPLIT FUNCTION
def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

sentences = []
labels = []

for _, row in data.iterrows():
    abstract = row['Abstract']
    summary = row['Summarized Abstract']
    abs_sentences = split_sentences(abstract)

    for sent in abs_sentences:
        sentences.append(sent)

        # Same labeling logic as your CNN
        if sent.strip().lower() in summary.lower():
            labels.append(1)
        else:
            labels.append(0)

sentences = np.array(sentences)
labels = np.array(labels)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=0.25, random_state=42
)

# TOKENIZATION
num_words = 10000
max_len = 60

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# CNN + RNN MODEL
model = Sequential([
    Embedding(input_dim=num_words, output_dim=128),

    Conv1D(128, 5, activation='relu', padding='same'),
    LSTM(128, return_sequences=True),

    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# TRAIN MODEL
model.fit(
    X_train_pad,
    y_train,
    epochs=8,
    batch_size=32,
    validation_split=0.2
)

# SUMMARY GENERATION FUNCTION
def generate_summary(text, top_k=3):
    sents = split_sentences(text)
    seq = tokenizer.texts_to_sequences(sents)
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    scores = model.predict(pad, verbose=0).flatten()

    top_indices = scores.argsort()[-top_k:][::-1]
    selected = [sents[i] for i in sorted(top_indices)]

    return " ".join(selected)

# PRINT ACTUAL AND PREDICTED SUMMARIES
for i in range(5):
    original = data.iloc[i]['Abstract']
    actual = data.iloc[i]['Summarized Abstract']
    predicted = generate_summary(original)

    print("\nExample", i+1)
    print("Actual Summary:", actual)
    print("Predicted Summary:", predicted)

# ROUGE Evaluation
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

test_sample = data.sample(frac=0.25, random_state=42)

for _, row in test_sample.iterrows():
    pred = generate_summary(row['Abstract'])
    ref = row['Summarized Abstract']

    scores = scorer.score(ref, pred)

    rouge1.append(scores['rouge1'].fmeasure)
    rouge2.append(scores['rouge2'].fmeasure)
    rougeL.append(scores['rougeL'].fmeasure)

print("\nROUGE-1:", np.mean(rouge1))
print("ROUGE-2:", np.mean(rouge2))
print("ROUGE-L:", np.mean(rougeL))

Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_32 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_16 (Conv1D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_24 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_11         │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 13s 230ms/step - accuracy: 0.8688 - loss: 0.4957 - val_accuracy: 1.0000 - val_loss: 0.0059
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 118ms/step - accuracy: 0.9982 - loss: 0.0155 - val_accuracy: 1.0000 - val_loss: 2.3914e-04
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 136ms/step - accuracy: 0.9996 - loss: 0.0042 - val_accuracy: 1.0000 - val_loss: 1.4725e-04
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - accuracy: 0.9974 - loss: 0.0252 - val_accuracy: 1.0000 - val_loss: 7.4605e-04
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 113ms/step - accuracy: 0.9978 - loss: 0.0140 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 111ms/step - accuracy: 0.9974 - loss: 0.0227 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 113ms/step - accuracy: 0.9961 - loss: 0.0293 - val_accuracy: 1.0000 - val_loss: 0.0020
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 140ms/step - accuracy: 0.9990 - loss: 0.0093 - val_accurac